In [1]:

# ============================================================
# CNN FACIAL EMOTION RECOGNITION PROJECT
# Classes: Angry, Happy, Neutral, Sad
# Framework: PyTorch + OpenCV
# ============================================================

import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# ------------------------------------------------------------
# 1. SETTINGS
# ------------------------------------------------------------

DATASET_PATH = "Emotion_Dataset"
MODEL_PATH = "emotion_cnn.pth"

IMG_SIZE = 48
BATCH_SIZE = 32
EPOCHS = 15
LEARNING_RATE = 0.001

# First run: "train"
# After training: "webcam"
MODE = "train"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


# ------------------------------------------------------------
# 2. IMAGE PREPROCESSING
# ------------------------------------------------------------

train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

val_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])


# ------------------------------------------------------------
# 3. LOAD DATASET
# ------------------------------------------------------------

def load_dataset():

    train_path = os.path.join(DATASET_PATH, "train")
    val_path = os.path.join(DATASET_PATH, "val")

    if not os.path.isdir(train_path):
        raise FileNotFoundError(
            "Training folder not found: " + train_path
        )

    if not os.path.isdir(val_path):
        raise FileNotFoundError(
            "Validation folder not found: " + val_path
        )

    train_data = datasets.ImageFolder(
        train_path,
        transform=train_transform
    )

    val_data = datasets.ImageFolder(
        val_path,
        transform=val_transform
    )

    if train_data.classes != val_data.classes:
        raise ValueError(
            "Training and validation classes do not match."
        )

    train_loader = DataLoader(
        train_data,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0
    )

    val_loader = DataLoader(
        val_data,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0
    )

    print("Emotion classes:", train_data.classes)
    print("Training images:", len(train_data))
    print("Validation images:", len(val_data))

    return train_loader, val_loader, train_data.classes


# ------------------------------------------------------------
# 4. BUILD CNN MODEL
# ------------------------------------------------------------

class EmotionCNN(nn.Module):

    def __init__(self, num_classes=4):

        super(EmotionCNN, self).__init__()

        self.features = nn.Sequential(

            # First convolution layer
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Second convolution layer
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Third convolution layer
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):

        x = self.features(x)
        x = self.classifier(x)

        return x


# ------------------------------------------------------------
# 5. TRAIN CNN MODEL
# ------------------------------------------------------------

def train_model():

    train_loader, val_loader, classes = load_dataset()

    model = EmotionCNN(
        num_classes=len(classes)
    ).to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    train_losses = []
    val_accuracies = []

    for epoch in range(EPOCHS):

        # Training
        model.train()

        running_loss = 0
        correct = 0
        total = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_accuracy = 100 * correct / total

        average_loss = running_loss / len(train_loader)

        # Validation
        model.eval()

        val_correct = 0
        val_total = 0

        with torch.no_grad():

            for images, labels in val_loader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)

                _, predicted = torch.max(outputs, 1)

                val_total += labels.size(0)

                val_correct += (
                    predicted == labels
                ).sum().item()

        val_accuracy = 100 * val_correct / val_total

        train_losses.append(average_loss)
        val_accuracies.append(val_accuracy)

        print(
            f"Epoch [{epoch+1}/{EPOCHS}] "
            f"Loss: {average_loss:.4f} "
            f"Train Accuracy: {train_accuracy:.2f}% "
            f"Validation Accuracy: {val_accuracy:.2f}%"
        )

    # Save trained model
    torch.save({
        "model_state_dict": model.state_dict(),
        "classes": classes
    }, MODEL_PATH)

    print("\nModel successfully saved:", MODEL_PATH)

    # Plot training loss
    plt.figure(figsize=(8, 4))
    plt.plot(train_losses, label="Training Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("CNN Training Loss")
    plt.legend()
    plt.show()

    # Plot validation accuracy
    plt.figure(figsize=(8, 4))
    plt.plot(val_accuracies, label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.title("CNN Validation Accuracy")
    plt.legend()
    plt.show()


# ------------------------------------------------------------
# 6. REAL-TIME WEBCAM EMOTION DETECTION
# ------------------------------------------------------------

def webcam_detection():

    if not os.path.exists(MODEL_PATH):
        print("Model not found. Train the model first.")
        return

    # Load saved model
    checkpoint = torch.load(
        MODEL_PATH,
        map_location=device,
        weights_only=True
    )

    classes = checkpoint["classes"]

    model = EmotionCNN(
        num_classes=len(classes)
    ).to(device)

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    model.eval()

    # Face detection using OpenCV Haar Cascade
    face_detector = cv2.CascadeClassifier(
        cv2.data.haarcascades +
        "haarcascade_frontalface_default.xml"
    )

    if face_detector.empty():
        print("Face detector could not be loaded.")
        return

    # Camera
    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print("Webcam could not be opened.")
        return

    print("Webcam started. Press Q to exit.")

    while True:

        ret, frame = cap.read()

        if not ret:
            print("Cannot read webcam frame.")
            break

        gray = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2GRAY
        )

        faces = face_detector.detectMultiScale(
            gray,
            scaleFactor=1.2,
            minNeighbors=5,
            minSize=(40, 40)
        )

        for (x, y, w, h) in faces:

            # Crop face
            face = gray[y:y+h, x:x+w]

            # Convert face into CNN input
            face_image = Image.fromarray(face)

            input_transform = transforms.Compose([
                transforms.Resize((IMG_SIZE, IMG_SIZE)),
                transforms.ToTensor(),
                transforms.Normalize((0.5,), (0.5,))
            ])

            face_tensor = input_transform(
                face_image
            ).unsqueeze(0).to(device)

            # Predict emotion
            with torch.no_grad():

                output = model(face_tensor)

                probabilities = torch.softmax(
                    output, dim=1
                )

                confidence, prediction = torch.max(
                    probabilities, 1
                )

            emotion = classes[prediction.item()]

            confidence_value = confidence.item() * 100

            # Draw face rectangle
            cv2.rectangle(
                frame,
                (x, y),
                (x+w, y+h),
                (0, 255, 0),
                2
            )

            label = (
                f"{emotion.upper()} "
                f"{confidence_value:.1f}%"
            )

            cv2.putText(
                frame,
                label,
                (x, max(25, y-10)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2
            )

        cv2.imshow(
            "CNN Facial Emotion Recognition",
            frame
        )

        # Press Q to close webcam
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()


# ------------------------------------------------------------
# 7. RUN PROJECT
# ------------------------------------------------------------

if MODE == "train":

    train_model()

elif MODE == "webcam":

    webcam_detection()

else:

    print("Please select train or webcam mode.")


Using device: cpu


FileNotFoundError: Training folder not found: Emotion_Dataset\train

In [3]:

import os

# Create dataset folders automatically

folders = [
    "Emotion_Dataset/train/angry",
    "Emotion_Dataset/train/happy",
    "Emotion_Dataset/train/neutral",
    "Emotion_Dataset/train/sad",

    "Emotion_Dataset/val/angry",
    "Emotion_Dataset/val/happy",
    "Emotion_Dataset/val/neutral",
    "Emotion_Dataset/val/sad"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("All dataset folders created successfully!")

print("\nDataset location:")
print(os.path.abspath("Emotion_Dataset"))


All dataset folders created successfully!

Dataset location:
C:\Users\asus\Emotion_Dataset


In [5]:

import os

DATASET_PATH = "Emotion_Dataset"

for split in ["train", "val"]:
    print("\n", split.upper())

    for emotion in ["angry", "happy", "neutral", "sad"]:

        path = os.path.join(
            DATASET_PATH, split, emotion
        )

        if os.path.exists(path):
            images = [
                f for f in os.listdir(path)
                if f.lower().endswith(
                    (".jpg", ".jpeg", ".png")
                )
            ]

            print(emotion, ":", len(images), "images")

        else:
            print(emotion, ": Folder missing")



 TRAIN
angry : 0 images
happy : 0 images
neutral : 0 images
sad : 0 images

 VAL
angry : 0 images
happy : 0 images
neutral : 0 images
sad : 0 images


In [7]:
MODE = "train"

In [9]:

# ============================================================
# CNN FACIAL EMOTION RECOGNITION PROJECT
# Classes: Angry, Happy, Neutral, Sad
# Framework: PyTorch + OpenCV
# ============================================================

import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# ------------------------------------------------------------
# 1. SETTINGS
# ------------------------------------------------------------

DATASET_PATH = "Emotion_Dataset"
MODEL_PATH = "emotion_cnn.pth"

IMG_SIZE = 48
BATCH_SIZE = 32
EPOCHS = 15
LEARNING_RATE = 0.001

# First run: "train"
# After training: "webcam"
MODE = "train"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


# ------------------------------------------------------------
# 2. IMAGE PREPROCESSING
# ------------------------------------------------------------

train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

val_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])


# ------------------------------------------------------------
# 3. LOAD DATASET
# ------------------------------------------------------------

def load_dataset():

    train_path = os.path.join(DATASET_PATH, "train")
    val_path = os.path.join(DATASET_PATH, "val")

    if not os.path.isdir(train_path):
        raise FileNotFoundError(
            "Training folder not found: " + train_path
        )

    if not os.path.isdir(val_path):
        raise FileNotFoundError(
            "Validation folder not found: " + val_path
        )

    train_data = datasets.ImageFolder(
        train_path,
        transform=train_transform
    )

    val_data = datasets.ImageFolder(
        val_path,
        transform=val_transform
    )

    if train_data.classes != val_data.classes:
        raise ValueError(
            "Training and validation classes do not match."
        )

    train_loader = DataLoader(
        train_data,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0
    )

    val_loader = DataLoader(
        val_data,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0
    )

    print("Emotion classes:", train_data.classes)
    print("Training images:", len(train_data))
    print("Validation images:", len(val_data))

    return train_loader, val_loader, train_data.classes


# ------------------------------------------------------------
# 4. BUILD CNN MODEL
# ------------------------------------------------------------

class EmotionCNN(nn.Module):

    def __init__(self, num_classes=4):

        super(EmotionCNN, self).__init__()

        self.features = nn.Sequential(

            # First convolution layer
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Second convolution layer
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Third convolution layer
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):

        x = self.features(x)
        x = self.classifier(x)

        return x


# ------------------------------------------------------------
# 5. TRAIN CNN MODEL
# ------------------------------------------------------------

def train_model():

    train_loader, val_loader, classes = load_dataset()

    model = EmotionCNN(
        num_classes=len(classes)
    ).to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    train_losses = []
    val_accuracies = []

    for epoch in range(EPOCHS):

        # Training
        model.train()

        running_loss = 0
        correct = 0
        total = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_accuracy = 100 * correct / total

        average_loss = running_loss / len(train_loader)

        # Validation
        model.eval()

        val_correct = 0
        val_total = 0

        with torch.no_grad():

            for images, labels in val_loader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)

                _, predicted = torch.max(outputs, 1)

                val_total += labels.size(0)

                val_correct += (
                    predicted == labels
                ).sum().item()

        val_accuracy = 100 * val_correct / val_total

        train_losses.append(average_loss)
        val_accuracies.append(val_accuracy)

        print(
            f"Epoch [{epoch+1}/{EPOCHS}] "
            f"Loss: {average_loss:.4f} "
            f"Train Accuracy: {train_accuracy:.2f}% "
            f"Validation Accuracy: {val_accuracy:.2f}%"
        )

    # Save trained model
    torch.save({
        "model_state_dict": model.state_dict(),
        "classes": classes
    }, MODEL_PATH)

    print("\nModel successfully saved:", MODEL_PATH)

    # Plot training loss
    plt.figure(figsize=(8, 4))
    plt.plot(train_losses, label="Training Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("CNN Training Loss")
    plt.legend()
    plt.show()

    # Plot validation accuracy
    plt.figure(figsize=(8, 4))
    plt.plot(val_accuracies, label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.title("CNN Validation Accuracy")
    plt.legend()
    plt.show()


# ------------------------------------------------------------
# 6. REAL-TIME WEBCAM EMOTION DETECTION
# ------------------------------------------------------------

def webcam_detection():

    if not os.path.exists(MODEL_PATH):
        print("Model not found. Train the model first.")
        return

    # Load saved model
    checkpoint = torch.load(
        MODEL_PATH,
        map_location=device,
        weights_only=True
    )

    classes = checkpoint["classes"]

    model = EmotionCNN(
        num_classes=len(classes)
    ).to(device)

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    model.eval()

    # Face detection using OpenCV Haar Cascade
    face_detector = cv2.CascadeClassifier(
        cv2.data.haarcascades +
        "haarcascade_frontalface_default.xml"
    )

    if face_detector.empty():
        print("Face detector could not be loaded.")
        return

    # Camera
    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print("Webcam could not be opened.")
        return

    print("Webcam started. Press Q to exit.")

    while True:

        ret, frame = cap.read()

        if not ret:
            print("Cannot read webcam frame.")
            break

        gray = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2GRAY
        )

        faces = face_detector.detectMultiScale(
            gray,
            scaleFactor=1.2,
            minNeighbors=5,
            minSize=(40, 40)
        )

        for (x, y, w, h) in faces:

            # Crop face
            face = gray[y:y+h, x:x+w]

            # Convert face into CNN input
            face_image = Image.fromarray(face)

            input_transform = transforms.Compose([
                transforms.Resize((IMG_SIZE, IMG_SIZE)),
                transforms.ToTensor(),
                transforms.Normalize((0.5,), (0.5,))
            ])

            face_tensor = input_transform(
                face_image
            ).unsqueeze(0).to(device)

            # Predict emotion
            with torch.no_grad():

                output = model(face_tensor)

                probabilities = torch.softmax(
                    output, dim=1
                )

                confidence, prediction = torch.max(
                    probabilities, 1
                )

            emotion = classes[prediction.item()]

            confidence_value = confidence.item() * 100

            # Draw face rectangle
            cv2.rectangle(
                frame,
                (x, y),
                (x+w, y+h),
                (0, 255, 0),
                2
            )

            label = (
                f"{emotion.upper()} "
                f"{confidence_value:.1f}%"
            )

            cv2.putText(
                frame,
                label,
                (x, max(25, y-10)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2
            )

        cv2.imshow(
            "CNN Facial Emotion Recognition",
            frame
        )

        # Press Q to close webcam
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()


# ------------------------------------------------------------
# 7. RUN PROJECT
# ------------------------------------------------------------

if MODE == "train":

    train_model()

elif MODE == "webcam":

    webcam_detection()

else:

    print("Please select train or webcam mode.")


Using device: cpu


FileNotFoundError: Found no valid file for the classes angry, happy, neutral, sad. Supported extensions are: .jpg, .jpeg, .png, .ppm, .bmp, .pgm, .tif, .tiff, .webp

In [11]:
pip install opencv-python deepface tf-keras

In [13]:

import cv2
from deepface import DeepFace

# Start webcam
cap = cv2.VideoCapture(0)

print("Emotion detection started")
print("Press Q to exit")

while True:

    ret, frame = cap.read()

    if not ret:
        print("Webcam error")
        break

    try:
        # Detect emotion
        result = DeepFace.analyze(
            frame,
            actions=['emotion'],
            enforce_detection=False,
            detector_backend='opencv',
            silent=True
        )

        if isinstance(result, list):
            result = result[0]

        emotion = result['dominant_emotion']
        score = result['emotion'][emotion]

        # Display emotion
        cv2.putText(
            frame,
            emotion.upper(),
            (30, 50),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            f"Confidence: {score:.1f}%",
            (30, 90),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 0),
            2
        )

    except Exception as e:
        cv2.putText(
            frame,
            "Face not detected",
            (30, 50),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 0, 255),
            2
        )

    cv2.imshow("Facial Emotion Recognition", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


AttributeError: module 'tensorflow' has no attribute '__version__'

In [15]:
import torch
import cv2

print("PyTorch:", torch.__version__)
print("OpenCV:", cv2.__version__)

PyTorch: 2.14.0+cpu
OpenCV: 4.11.0


In [17]:
import sys
print("Python:", sys.version)

try:
    import torch
    print("PyTorch:", torch.__version__)
except Exception as e:
    print("PyTorch error:", e)

try:
    import cv2
    print("OpenCV:", cv2.__version__)
except Exception as e:
    print("OpenCV error:", e)

try:
    import tensorflow as tf
    print("TensorFlow:", getattr(tf, "__version__", "Version missing"))
    print("TensorFlow location:", getattr(tf, "__file__", "Unknown"))
except Exception as e:
    print("TensorFlow error:", e)

Python: 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 13:17:27) [MSC v.1929 64 bit (AMD64)]
PyTorch: 2.14.0+cpu
OpenCV: 4.11.0
TensorFlow: Version missing
TensorFlow location: None


In [19]:

# ==========================================================
# CNN FACIAL EMOTION RECOGNITION
# HAPPY | SAD | ANGRY | NEUTRAL
# Python + PyTorch + OpenCV
# ==========================================================

import os
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# ==========================================================
# 1. SETTINGS
# ==========================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

IMAGE_SIZE = 48
BATCH_SIZE = 32
EPOCHS = 10
MODEL_FILE = "emotion_cnn.pth"

EMOTIONS = ["angry", "happy", "neutral", "sad"]

print("Device:", DEVICE)


# ==========================================================
# 2. DOWNLOAD FER2013 DATASET
# ==========================================================

print("\nDownloading FER2013 dataset...")

dataset = load_dataset("zalando-datasets/fer2013")

print(dataset)


# ==========================================================
# 3. IMAGE PREPROCESSING
# ==========================================================

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])


# ==========================================================
# 4. PREPARE DATASET
# ==========================================================

# Standard FER2013 emotion labels
LABELS = {
    0: "angry",
    1: "disgust",
    2: "fear",
    3: "happy",
    4: "sad",
    5: "surprise",
    6: "neutral"
}


class EmotionDataset(Dataset):

    def __init__(self, data):

        self.images = []
        self.labels = []

        for item in data:

            label = LABELS[int(item["label"])]

            if label in EMOTIONS:

                self.images.append(item["image"])

                self.labels.append(
                    EMOTIONS.index(label)
                )

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):

        image = self.images[index].convert("RGB")

        image = transform(image)

        label = self.labels[index]

        return image, label


print("\nPreparing training data...")

train_data = EmotionDataset(dataset["train"])

test_data = EmotionDataset(dataset["test"])

train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

test_loader = DataLoader(
    test_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Training images:", len(train_data))
print("Testing images:", len(test_data))


# ==========================================================
# 5. CREATE CNN MODEL
# ==========================================================

class EmotionCNN(nn.Module):

    def __init__(self):

        super(EmotionCNN, self).__init__()

        self.conv_layers = nn.Sequential(

            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc_layers = nn.Sequential(

            nn.Flatten(),

            nn.Linear(128 * 6 * 6, 128),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(128, 4)
        )

    def forward(self, x):

        x = self.conv_layers(x)

        x = self.fc_layers(x)

        return x


model = EmotionCNN().to(DEVICE)

print("\nCNN Model Created Successfully")


# ==========================================================
# 6. TRAIN CNN MODEL
# ==========================================================

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

train_losses = []
train_accuracies = []

print("\nStarting CNN Training...")

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(DEVICE)

        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()

        total += labels.size(0)

    accuracy = 100 * correct / total

    average_loss = total_loss / len(train_loader)

    train_losses.append(average_loss)

    train_accuracies.append(accuracy)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss: {average_loss:.4f} | "
        f"Accuracy: {accuracy:.2f}%"
    )


# ==========================================================
# 7. TEST CNN MODEL
# ==========================================================

model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(DEVICE)

        labels = labels.to(DEVICE)

        outputs = model(images)

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()

        total += labels.size(0)

test_accuracy = 100 * correct / total

print("\nTest Accuracy:", round(test_accuracy, 2), "%")


# ==========================================================
# 8. SAVE MODEL
# ==========================================================

torch.save(
    model.state_dict(),
    MODEL_FILE
)

print("\nModel saved:", MODEL_FILE)


# ==========================================================
# 9. DISPLAY TRAINING GRAPH
# ==========================================================

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)

plt.plot(train_losses)

plt.title("CNN Training Loss")

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.subplot(1, 2, 2)

plt.plot(train_accuracies)

plt.title("CNN Training Accuracy")

plt.xlabel("Epoch")

plt.ylabel("Accuracy (%)")

plt.tight_layout()

plt.show()


# ==========================================================
# 10. LOAD TRAINED MODEL
# ==========================================================

model.load_state_dict(
    torch.load(
        MODEL_FILE,
        map_location=DEVICE,
        weights_only=True
    )
)

model.eval()

print("\nTrained model loaded successfully!")


# ==========================================================
# 11. FACE DETECTION USING OPENCV
# ==========================================================

face_detector = cv2.CascadeClassifier(
    cv2.data.haarcascades +
    "haarcascade_frontalface_default.xml"
)

if face_detector.empty():

    raise RuntimeError("Face detector not loaded!")


# ==========================================================
# 12. WEBCAM EMOTION RECOGNITION
# ==========================================================

print("\nStarting webcam...")

cap = cv2.VideoCapture(0)

if not cap.isOpened():

    raise RuntimeError("Cannot open webcam!")


while True:

    ret, frame = cap.read()

    if not ret:
        print("Cannot read webcam frame")
        break

    gray = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2GRAY
    )

    faces = face_detector.detectMultiScale(
        gray,
        scaleFactor=1.2,
        minNeighbors=5,
        minSize=(40, 40)
    )

    for (x, y, w, h) in faces:

        face = gray[y:y+h, x:x+w]

        face_image = Image.fromarray(face)

        face_tensor = transform(
            face_image
        ).unsqueeze(0).to(DEVICE)

        # CNN prediction
        with torch.no_grad():

            output = model(face_tensor)

            probabilities = F.softmax(
                output,
                dim=1
            )

            confidence, prediction = torch.max(
                probabilities,
                dim=1
            )

        emotion = EMOTIONS[prediction.item()]

        confidence_value = confidence.item() * 100

        label = (
            emotion.upper() +
            " " +
            str(round(confidence_value, 1)) +
            "%"
        )

        # Draw face rectangle
        cv2.rectangle(
            frame,
            (x, y),
            (x+w, y+h),
            (0, 255, 0),
            2
        )

        # Display prediction
        cv2.putText(
            frame,
            label,
            (x, max(25, y-10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )

    cv2.imshow(
        "CNN Facial Emotion Recognition",
        frame
    )

    # Press Q to exit
    if cv2.waitKey(1) & 0xFF == ord("q"):

        break


# ==========================================================
# 13. RELEASE WEBCAM
# ==========================================================

cap.release()

cv2.destroyAllWindows()

print("\nProject Completed Successfully!")


ModuleNotFoundError: No module named 'datasets'

In [1]:
%pip install datasets

  Using cached datasets-5.2.0-py3-none-any.whl.metadata (24 kB)
Using cached datasets-5.2.0-py3-none-any.whl (608 kB)
Note: you may need to restart the kernel to use updated packages.


In [3]:
from datasets import load_dataset

print("Datasets library installed successfully!")

ImportError: pyarrow requires NumPy 2.0 or newer, found 1.26.4

In [5]:
%pip install torch opencv-python numpy

Note: you may need to restart the kernel to use updated packages.


In [9]:

import cv2
import torch
import torch.nn as nn
import numpy as np
import torch.nn.functional as F

# --------------------------------
# 1. CNN MODEL
# --------------------------------

class EmotionCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((4, 4))
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 4 * 4, 64),
            nn.ReLU(),
            nn.Linear(64, 4)
        )

    def forward(self, x):
        return self.fc(self.conv(x))


# --------------------------------
# 2. CREATE MODEL
# --------------------------------

device = torch.device("cpu")

model = EmotionCNN().to(device)

# Demonstration only: random weights.
# Train with real labelled images before using
# the predictions as emotion classifications.

model.eval()

emotions = ["Angry", "Happy", "Neutral", "Sad"]

print("CNN model created successfully!")


# --------------------------------
# 3. FACE DETECTION
# --------------------------------

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades +
    "haarcascade_frontalface_default.xml"
)

if face_cascade.empty():
    raise RuntimeError("Face detector not loaded")


# --------------------------------
# 4. START WEBCAM
# --------------------------------

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Webcam not available")

print("Webcam started. Press Q to exit.")


# --------------------------------
# 5. REAL-TIME FACE DETECTION
# --------------------------------

while True:

    ret, frame = cap.read()

    if not ret:
        break

    gray = cv2.cvtColor(
        frame, cv2.COLOR_BGR2GRAY
    )

    faces = face_cascade.detectMultiScale(
        gray,
        1.2,
        5,
        minSize=(40, 40)
    )

    for (x, y, w, h) in faces:

        face = gray[y:y+h, x:x+w]

        face = cv2.resize(face, (48, 48))

        face = face.astype("float32") / 255.0

        tensor = torch.tensor(
            face,
            dtype=torch.float32
        ).unsqueeze(0).unsqueeze(0)

        with torch.no_grad():
            output = model(tensor)

            prediction = torch.argmax(
                output, dim=1
            ).item()

        emotion = emotions[prediction]

        cv2.rectangle(
            frame,
            (x, y),
            (x+w, y+h),
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            "Face Detected",
            (x, y-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )

    cv2.imshow(
        "Simple CNN Face Detection",
        frame
    )

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


# --------------------------------
# 6. CLOSE WEBCAM
# --------------------------------

cap.release()
cv2.destroyAllWindows()

print("Project completed!")


CNN model created successfully!
Webcam started. Press Q to exit.
Project completed!


In [13]:

# ==========================================================
# OPTION A: SIMPLE YOLO HUMAN DETECTION SYSTEM
# Person + Face + Eyes + Smile + Optional Age/Gender/Gestures
# ==========================================================

import os
import cv2
import torch
from ultralytics import YOLO

# ---------------- SETTINGS ----------------

DEVICE = "cpu"

YOLO_MODEL = "yolo11n.pt"

# Optional custom YOLO gesture model
GESTURE_MODEL_PATH = "gesture_best.pt"

# Optional OpenCV age/gender models
AGE_MODEL = "age_net.caffemodel"
AGE_CONFIG = "age_deploy.prototxt"
GENDER_MODEL = "gender_net.caffemodel"
GENDER_CONFIG = "gender_deploy.prototxt"

print("Loading YOLO model...")

model = YOLO(YOLO_MODEL)

# Load face, eye and smile detectors
cascade_path = cv2.data.haarcascades

face_detector = cv2.CascadeClassifier(
    cascade_path + "haarcascade_frontalface_default.xml"
)

eye_detector = cv2.CascadeClassifier(
    cascade_path + "haarcascade_eye.xml"
)

smile_detector = cv2.CascadeClassifier(
    cascade_path + "haarcascade_smile.xml"
)

if face_detector.empty():
    raise RuntimeError("Face detector failed to load")

if eye_detector.empty() or smile_detector.empty():
    raise RuntimeError("Eye or smile detector failed to load")

# Load optional gesture model
gesture_model = None

if os.path.exists(GESTURE_MODEL_PATH):
    gesture_model = YOLO(GESTURE_MODEL_PATH)
    print("Gesture model loaded")
else:
    print("Gesture model not found. Gesture detection disabled.")

# Load optional age/gender models
age_gender_net = None

age_files = [
    AGE_MODEL, AGE_CONFIG, GENDER_MODEL, GENDER_CONFIG
]

if all(os.path.exists(f) for f in age_files):

    age_net = cv2.dnn.readNet(AGE_MODEL, AGE_CONFIG)
    gender_net = cv2.dnn.readNet(GENDER_MODEL, GENDER_CONFIG)

    age_gender_net = True

    print("Age and gender models loaded")

else:
    age_net = None
    gender_net = None
    print("Age/gender model files missing. Estimation disabled.")

# Age/gender model categories
AGE_BUCKETS = [
    "(0-2)", "(4-6)", "(8-12)", "(15-20)",
    "(25-32)", "(38-43)", "(48-53)", "(60-100)"
]

GENDER_LABELS = ["Female", "Male"]

# ---------------- WEBCAM ----------------

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Webcam not available")

print("YOLO project started. Press Q to exit.")

while True:

    ret, frame = cap.read()

    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # YOLO person detection
    results = model.predict(
        frame,
        imgsz=640,
        conf=0.4,
        device=DEVICE,
        verbose=False
    )

    for result in results:

        for box in result.boxes:

            class_id = int(box.cls[0])

            if model.names[class_id] != "person":
                continue

            x1, y1, x2, y2 = map(
                int, box.xyxy[0].tolist()
            )

            cv2.rectangle(
                frame, (x1, y1), (x2, y2),
                (255, 0, 0), 2
            )

            cv2.putText(
                frame, "PERSON",
                (x1, max(20, y1-10)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6, (255, 0, 0), 2
            )

    # Face detection
    faces = face_detector.detectMultiScale(
        gray,
        scaleFactor=1.2,
        minNeighbors=5,
        minSize=(50, 50)
    )

    for (x, y, w, h) in faces:

        face_gray = gray[y:y+h, x:x+w]
        face_color = frame[y:y+h, x:x+w]

        cv2.rectangle(
            frame, (x, y), (x+w, y+h),
            (0, 255, 0), 2
        )

        cv2.putText(
            frame, "FACE",
            (x, max(20, y-10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6, (0, 255, 0), 2
        )

        # Eye detection
        eyes = eye_detector.detectMultiScale(
            face_gray,
            scaleFactor=1.1,
            minNeighbors=6,
            minSize=(10, 10)
        )

        for (ex, ey, ew, eh) in eyes:

            cv2.rectangle(
                frame,
                (x+ex, y+ey),
                (x+ex+ew, y+ey+eh),
                (0, 255, 255), 2
            )

            cv2.putText(
                frame, "EYE",
                (x+ex, y+ey-5),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.4, (0, 255, 255), 1
            )

        # Smile detection
        smiles = smile_detector.detectMultiScale(
            face_gray,
            scaleFactor=1.7,
            minNeighbors=22,
            minSize=(20, 10)
        )

        if len(smiles) > 0:

            cv2.putText(
                frame, "SMILE",
                (x, y+h+20),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6, (0, 255, 255), 2
            )

        # Age and gender estimation
        if age_gender_net is not None:

            blob = cv2.dnn.blobFromImage(
                face_color,
                1.0,
                (227, 227),
                (78.426, 87.769, 114.896),
                swapRB=False
            )

            gender_net.setInput(blob)
            gender_prediction = gender_net.forward()

            age_net.setInput(blob)
            age_prediction = age_net.forward()

            gender = GENDER_LABELS[
                gender_prediction[0].argmax()
            ]

            age = AGE_BUCKETS[
                age_prediction[0].argmax()
            ]

            cv2.putText(
                frame,
                f"Estimated age: {age}",
                (x, y+h+40),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5, (255, 255, 0), 2
            )

            cv2.putText(
                frame,
                f"Model presentation: {gender}",
                (x, y+h+60),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5, (255, 255, 0), 2
            )

    # Optional hand gesture YOLO detection
    if gesture_model is not None:

        gesture_results = gesture_model.predict(
            frame,
            conf=0.4,
            device=DEVICE,
            verbose=False
        )

        for result in gesture_results:

            for box in result.boxes:

                x1, y1, x2, y2 = map(
                    int, box.xyxy[0].tolist()
                )

                gesture = gesture_model.names[
                    int(box.cls[0])
                ]

                cv2.rectangle(
                    frame, (x1, y1), (x2, y2),
                    (0, 165, 255), 2
                )

                cv2.putText(
                    frame, "GESTURE: " + gesture,
                    (x1, max(20, y1-10)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, (0, 165, 255), 2
                )

    cv2.imshow("YOLO Human Detection System", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

print("Project completed")


Loading YOLO model...
Gesture model not found. Gesture detection disabled.
Age/gender model files missing. Estimation disabled.
YOLO project started. Press Q to exit.
Project completed


In [15]:

# ==========================================================
# ADVANCED PROJECT: CNN AGE AND GENDER MODEL TRAINING
# ==========================================================

import os
import cv2
import torch
import torch.nn as nn
import numpy as np

from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

# Settings
UTKFACE_PATH = "UTKFace"

DEVICE = torch.device("cpu")

BATCH_SIZE = 32
EPOCHS = 5

IMAGE_SIZE = 64

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.5, 0.5, 0.5),
        (0.5, 0.5, 0.5)
    )
])


# Dataset
class AgeGenderDataset(Dataset):

    def __init__(self, folder):

        self.samples = []

        if not os.path.isdir(folder):
            raise FileNotFoundError(
                "UTKFace folder not found: " + folder
            )

        for filename in os.listdir(folder):

            if not filename.lower().endswith(
                (".jpg", ".jpeg", ".png")
            ):
                continue

            try:

                parts = filename.split("_")

                age = int(parts[0])
                gender = int(parts[1])

                if not 0 <= age <= 116:
                    continue

                if gender not in (0, 1):
                    continue

                self.samples.append(
                    (os.path.join(folder, filename),
                     age, gender)
                )

            except (ValueError, IndexError):
                continue

        if not self.samples:
            raise RuntimeError(
                "No valid UTKFace images found."
            )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):

        path, age, gender = self.samples[index]

        image = Image.open(path).convert("RGB")

        image = transform(image)

        return (
            image,
            torch.tensor(age, dtype=torch.float32),
            torch.tensor(gender, dtype=torch.long)
        )


# Load data
dataset = AgeGenderDataset(UTKFACE_PATH)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_data, val_data = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Training images:", len(train_data))
print("Validation images:", len(val_data))


# CNN architecture
class AgeGenderCNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 128, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),

            nn.Flatten()
        )

        self.shared = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        self.age_head = nn.Linear(64, 1)

        self.gender_head = nn.Linear(64, 2)

    def forward(self, x):

        x = self.features(x)

        x = self.shared(x)

        age = self.age_head(x).squeeze(1)

        gender = self.gender_head(x)

        return age, gender


model = AgeGenderCNN().to(DEVICE)

age_loss_fn = nn.L1Loss()

gender_loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)


# Training
for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for images, ages, genders in train_loader:

        images = images.to(DEVICE)
        ages = ages.to(DEVICE)
        genders = genders.to(DEVICE)

        optimizer.zero_grad()

        predicted_age, predicted_gender = model(images)

        age_loss = age_loss_fn(
            predicted_age, ages
        )

        gender_loss = gender_loss_fn(
            predicted_gender, genders
        )

        loss = age_loss + gender_loss

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{EPOCHS} "
        f"Loss: {total_loss/len(train_loader):.4f}"
    )


# Validation
model.eval()

total_age_error = 0
gender_correct = 0
total = 0

with torch.no_grad():

    for images, ages, genders in val_loader:

        images = images.to(DEVICE)
        ages = ages.to(DEVICE)
        genders = genders.to(DEVICE)

        predicted_age, predicted_gender = model(images)

        total_age_error += torch.abs(
            predicted_age - ages
        ).sum().item()

        predictions = predicted_gender.argmax(dim=1)

        gender_correct += (
            predictions == genders
        ).sum().item()

        total += len(ages)

print(
    "Mean absolute age error:",
    round(total_age_error / total, 2)
)

print(
    "Gender-label validation accuracy:",
    round(100 * gender_correct / total, 2), "%"
)


# Save
torch.save(
    model.state_dict(),
    "age_gender_cnn.pth"
)

print("Age and gender CNN saved successfully!")


FileNotFoundError: UTKFace folder not found: UTKFace

In [17]:
%pip install ultralytics opencv-python

Note: you may need to restart the kernel to use updated packages.


In [ ]:

import cv2
from ultralytics import YOLO

# Load YOLO pretrained model
model = YOLO("yolo11n.pt")

# Open webcam
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Webcam not found!")
else:
    print("YOLO Object Detection Started")
    print("Press Q to stop")

    while True:
        ret, frame = cap.read()

        if not ret:
            print("Cannot read webcam frame")
            break

        # Detect objects
        results = model.predict(
            source=frame,
            conf=0.5,
            imgsz=320,
            device="cpu",
            verbose=False
        )

        # Draw detection boxes and labels
        output_frame = results[0].plot()

        # Display webcam
        cv2.imshow("YOLO Real-Time Object Detection", output_frame)

        # Press Q to exit
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()
